# Seto — Full Training Pipeline

## Stages
1. **Pretrain** — next-token prediction on web data (50-300B tokens)
2. **Cooldown** — higher quality data, lower LR (last 5-15%)
3. **SFT** — instruction following on chat data (200k-2M samples)
4. **DPO** — preference optimization (chosen vs rejected pairs)

## Setup
1. Upload `seto/` as Kaggle dataset
2. Add training data via **Add Data**
3. Configure paths in cells below
4. Run stages in order

In [ ]:
!pip install -q tokenizers datasets accelerate tqdm

In [ ]:
import torch, os, sys, shutil, glob, json
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_mem/1e9:.1f}GB)")
print(f"bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Copy seto to working dir
SETO_SRC = "/kaggle/input/seto-model"
WORKING = "/kaggle/working"

if os.path.exists(f"{SETO_SRC}/seto") and not os.path.exists(f"{WORKING}/seto"):
    shutil.copytree(f"{SETO_SRC}/seto", f"{WORKING}/seto")
os.chdir(WORKING)
sys.path.insert(0, WORKING)
print(f"Ready: {os.listdir('seto/')}")

In [ ]:
# Find all datasets
KAGGLE_INPUT = "/kaggle/input"
datasets_found = []
for d in sorted(os.listdir(KAGGLE_INPUT)):
    p = os.path.join(KAGGLE_INPUT, d)
    if os.path.isdir(p):
        files = (glob.glob(f"{p}/**/*.jsonl", recursive=True) +
                 glob.glob(f"{p}/**/*.json", recursive=True) +
                 glob.glob(f"{p}/**/*.txt", recursive=True))
        if files:
            datasets_found.append({"name": d, "path": p, "files": len(files)})

print("Datasets:")
for ds in datasets_found:
    print(f"  {ds['name']}: {ds['files']} files")

In [ ]:
from seto.config import ModelConfig, TrainConfig, MODEL_SMALL, MODEL_BASE
from seto.config import STAGE_PRETRAIN, STAGE_COOLDOWN, STAGE_SFT, STAGE_DPO
from seto.model import SetoLM
from seto.tokenizer import SetoTokenizer
from seto.data import PretrainDataset, SFTDataset, DPODataset
from seto.trainer import SetoTrainer
from seto.sft import SFTTrainer
from seto.dpo import DPOTrainer

# ============================================================
# CONFIGURE THESE PATHS
# ============================================================
PRETRAIN_DATA = datasets_found[0]["path"] if datasets_found else None
SFT_DATA = None       # Set to SFT dataset path if available
DPO_DATA = None       # Set to DPO preference data path if available
TOKENIZER_DIR = "/kaggle/working/seto-tokenizer"

print(f"Pretrain data: {PRETRAIN_DATA}")
print(f"SFT data: {SFT_DATA}")
print(f"DPO data: {DPO_DATA}")

In [ ]:
# Initialize model
USE_SMALL = True  # Set False for 1B model
model_config = MODEL_SMALL if USE_SMALL else MODEL_BASE

model = SetoLM(model_config)
print(f"Model: {model.count_parameters():,} params ({model.count_parameters()/1e6:.1f}M)")
print(f"Size: {sum(p.numel()*p.element_size() for p in model.parameters())/1e9:.2f} GB")

In [ ]:
# Train tokenizer if needed
if not os.path.exists(f"{TOKENIZER_DIR}/tokenizer.json"):
    from seto.tokenizer import SetoTokenizer
    tokenizer = SetoTokenizer(vocab_size=model_config.vocab_size)
    train_files = []
    if PRETRAIN_DATA:
        for ext in ["*.jsonl", "*.txt"]:
            train_files.extend(glob.glob(f"{PRETRAIN_DATA}/**/{ext}", recursive=True)[:5])
    if train_files:
        tokenizer.train(train_files, TOKENIZER_DIR)
else:
    print("Tokenizer already exists")

tokenizer = SetoTokenizer.from_pretrained(TOKENIZER_DIR)
print(f"Vocab: {len(tokenizer)}")

In [ ]:
# ============================================================
# STAGE 1: PRETRAINING
# ============================================================
# Recommended: 50-300B tokens for first serious run
# 5B tokens = sanity check, 100B+ = real training

config = STAGE_PRETRAIN
config.checkpoint_dir = "/kaggle/working/checkpoints_pretrain"

if PRETRAIN_DATA:
    dataset = PretrainDataset(PRETRAIN_DATA, seq_len=model_config.max_seq_len, tokenizer=tokenizer)
    trainer = SetoTrainer(model, dataset, config=config)
    trainer.train()
else:
    print("No pretrain data found — skipping")

In [ ]:
# ============================================================
# STAGE 2: COOLDOWN (higher quality data, lower LR)
# ============================================================
# Last 5-15% of training on curated data
# Skip if doing short pretraining run

SKIP_COOLDOWN = True  # Set False for full training runs

if not SKIP_COOLDOWN and PRETRAIN_DATA:
    config = STAGE_COOLDOWN
    config.checkpoint_dir = "/kaggle/working/checkpoints_cooldown"
    config.resume_from = "/kaggle/working/checkpoints_pretrain/best"
    
    dataset = PretrainDataset(PRETRAIN_DATA, seq_len=model_config.max_seq_len, tokenizer=tokenizer)
    trainer = SetoTrainer(model, dataset, config=config)
    if config.resume_from and os.path.exists(config.resume_from):
        trainer.resume(config.resume_from)
    trainer.train()
else:
    print("Cooldown skipped")

In [ ]:
# ============================================================
# STAGE 3: SFT (instruction following)
# ============================================================
# Needs: conversation data with system/user/assistant turns
# Format: {"messages": [{"role": "system", "content": ...}, ...]}
# Sources: SmolTalk, UltraChat, OpenHermes, or custom data

if SFT_DATA:
    config = STAGE_SFT
    config.checkpoint_dir = "/kaggle/working/checkpoints_sft"
    
    sft_dataset = SFTDataset(SFT_DATA, seq_len=model_config.max_seq_len, tokenizer=tokenizer)
    sft_trainer = SFTTrainer(model, sft_dataset, tokenizer, config)
    sft_trainer.train()
else:
    print("No SFT data — add dataset and set SFT_DATA path")
    print("Expected format: {\"messages\": [{\"role\": \"user\", \"content\": ...}, ...]}")

In [ ]:
# ============================================================
# STAGE 4: DPO (preference optimization)
# ============================================================
# Needs: preference pairs
# Format: {"prompt": ..., "chosen": ..., "rejected": ...}
# Can use: UltraFeedback, synthetic pairs from teacher model

if DPO_DATA:
    config = STAGE_DPO
    config.checkpoint_dir = "/kaggle/working/checkpoints_dpo"
    
    dpo_dataset = DPODataset(DPO_DATA, seq_len=model_config.max_seq_len, tokenizer=tokenizer)
    
    # Reference model = current model before DPO
    import copy
    ref_model = copy.deepcopy(model)
    for p in ref_model.parameters():
        p.requires_grad = False
    ref_model.eval()
    
    dpo_trainer = DPOTrainer(model, ref_model, dpo_dataset, tokenizer, config)
    dpo_trainer.train()
else:
    print("No DPO data — add preference pairs and set DPO_DATA path")
    print("Expected format: {\"prompt\": ..., \"chosen\": ..., \"rejected\": ...}")

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================
import zipfile

FINAL_DIR = "/kaggle/working/seto-final"
os.makedirs(FINAL_DIR, exist_ok=True)

# Save weights
state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
torch.save(state_dict, f"{FINAL_DIR}/model.pt")

# Save config
with open(f"{FINAL_DIR}/config.json", "w") as f:
    json.dump(model_config.__dict__, f, indent=2)

# Copy tokenizer
if os.path.exists(TOKENIZER_DIR):
    shutil.copytree(TOKENIZER_DIR, f"{FINAL_DIR}/tokenizer", dirs_exist_ok=True)

# Zip
ZIP = "/kaggle/working/seto-final.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(FINAL_DIR):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, FINAL_DIR))

print(f"Final model: {ZIP} ({os.path.getsize(ZIP)/1e6:.1f} MB)")

In [ ]:
# List all checkpoints
for ckpt_dir in ["checkpoints_pretrain", "checkpoints_cooldown", "checkpoints_sft", "checkpoints_dpo"]:
    full = f"/kaggle/working/{ckpt_dir}"
    if os.path.exists(full):
        zips = sorted(f for f in os.listdir(full) if f.endswith(".zip"))
        if zips:
            print(f"{ckpt_dir} ({len(zips)} checkpoints):")
            for z in zips:
                print(f"  {z} ({os.path.getsize(os.path.join(full, z))/1e6:.1f} MB)")

In [ ]:
# Quick inference test
model.eval()
device = next(model.parameters()).device

prompts = [
    "<|system|>\nYou are Seto, a helpful bilingual assistant.\n<|user|>\nWhat is 2+2?\n<|assistant|>\n",
    "<|system|>\nТы Seto, полезный помощник.\n<|user|>\nПривет, как дела?\n<|assistant|>\n",
]

for prompt in prompts:
    ids = torch.tensor([tokenizer.encode(prompt, add_bos=True, add_eos=False)], device=device)
    with torch.no_grad():
        for _ in range(150):
            logits, _ = model(ids)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            ids = torch.cat([ids, next_token], dim=-1)
    output = tokenizer.decode(ids[0].tolist(), skip_special_tokens=True)
    print(f"Prompt: {prompt[-30:]}")
    print(f"Output: {output[-200:]}")
    print()